# Render the Dreadnoughts Using the Game Data

In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

In [3]:
!mkdir -p PagedOutArticle

### Load the Character Sets

In [4]:
import re
from colors_map import *

def loadCharacterSet(charset_lines, ref_offset=0):
    character_set = {}
    charset_data = []
    char_ref = None
    for l in charset_lines:
        if "CHARACTER" in l:
            if charset_data:
                character_set[char_ref] = charset_data
            charset_data = []
            ref = int(l[61:63],16) + ref_offset
            char_ref = f"{ref:0{2}x}".upper()
             
        m = re.findall(r"[0-1]{8}",l)
        if not m:
            continue
        bits = m[0].strip()
        line_bits = []
        for i in range(0,7,2):
            bitpair = bits[i:i+2]
            line_bits += [bitpair]
            line_bits += [bitpair]
        charset_data += [line_bits]
    character_set[char_ref] = charset_data
    return character_set

charsets_file = "uridium/src/charset.asm"
input_file = open(charsets_file,'r')
charset_lines = input_file.readlines()
charset_lines = charset_lines[1439:]
base_character_set = loadCharacterSet(charset_lines)

character_files = [
    "uridium/src/surface1_charset.asm",
    "uridium/src/surface2_charset.asm",
    "uridium/src/surface3_charset.asm",
]
character_sets = []
for filename in character_files:
    charsets_file = filename
    input_file = open(charsets_file,'r')
    charset_lines = input_file.readlines()
    character_set = base_character_set | loadCharacterSet(charset_lines, ref_offset=0x80)
    character_sets += [character_set]


## Generate PNG images of the Character Sets

In [5]:
from PIL import Image, ImageColor
CHARACTER_COLS = 8
CHARACTER_ROWS = 8

def paintRawCharacter(character_set, character, colors, verticalExpand=False):
    multicol0, multicol1,color,color1 = colors
    
    verticalExpansion = 2 if verticalExpand else 1
    CHARACTER_COLS = 8
    CHARACTER_ROWS = 8 * verticalExpansion

    colormap = {
        "01": multicol0,
        "10": color,
        "11": multicol1,
        "00": color1,
    }
    
    if character not in character_set:
        print(character)
        return
    
    image_width = CHARACTER_COLS
    image_height = CHARACTER_ROWS
    img = Image.new( 'RGBA', (image_width+1, image_height+1))
    pixels = img.load()

    bit_array = character_set[character]
    if verticalExpand:
        expanded_bit_array = []
        for a in bit_array:
            expanded_bit_array += [a,a]
        bit_array = expanded_bit_array
    
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            if bit == "11":
                continue
            pixel_color = ImageColor.getrgb(c64_to_rgb[colormap[bit]])
            pixels[x,y] = pixel_color
    return img

### Get the Level Specific Color-Schemes and Character Set References

In [6]:
raw_level_color_schemes = """
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_YELLOW,M_ORANGE  ; Level 1
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_LTBLUE,M_BLUE    ; Level 4
        .BYTE M_GREEN,M_LTGREEN,M_ORANGE,M_LTBLUE,M_BLUE  ; Level 5
        .BYTE M_ORANGE,M_YELLOW,M_ORANGE,M_GRAY2,M_BLACK  ; Level 6
        .BYTE M_GRAY1,M_CYAN,M_LTGREEN,M_LTRED,M_RED      ; Level 7
        .BYTE M_BLACK,M_GRAY2,M_LTRED,M_LTBLUE,M_BLUE     ; Level 8
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_GRAY1,M_GRAY2,M_GRAY2,M_LTRED,M_RED       ; Level 10
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_BLUE,M_CYAN,M_ORANGE,M_GRAY2,M_GRAY1      ; Level 12
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_RED,M_LTRED,M_ORANGE,M_YELLOW,M_ORANGE    ; Level 15
"""
raw_level_color_schemes = [l[14:57].split(',') for l in raw_level_color_schemes.split('\n')][1:-1]
raw_level_color_schemes = [[x.strip() for x in l] for l in raw_level_color_schemes]
#raw_level_color_schemes = [v for l in raw_level_color_schemes for v in l]
#surface_ram = [temp_surface_ram[v:v+80] for v in range(0, len(temp_surface_ram), 80)]
#surface_ram = ["$"+x.upper() for x in flatten(surface_ram)]
level_colors = [None]
for l in raw_level_color_schemes:
    colors = (color_constants[l[1]], "c64_black", color_constants[l[0]], "c64_white")
    level_colors += [colors]

# The character sets each level uses.
level_charsets = [0,2,0,2,0,1,2,1,0,1,2,0,2,1,0,2]

level_names = [None, "Zinc","Lead","Copper","Silver","Iron","Gold","Platinum","Tungsten","Iridon","Kallisto","Tri-alloy","Quadmium","Ergonite","Galactium","Uridium"]

### Create PNG Images of the Level-Specific Charactersets

In [8]:
!mkdir -p PagedOutArticle/surface_characters

In [9]:
level_character_set_images = [None]
for i in range(1,16):
    charset_ref = level_charsets[i]
    colors = level_colors[i]
    character_set = character_sets[charset_ref]
    character_images = {}
    for character_name in character_set:
        img = paintRawCharacter(character_set, character_name, colors)
        character_images["$"+character_name] = img
        if character_name:
            img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
            img.save(f"PagedOutArticle/surface_characters/{i}_{character_name}.png")
    level_character_set_images += [character_images]

In [10]:
def generateMiniCharDiagram(char_img, char_name):
    image_width = 10 + char_img.width + 10
    image_height = 10 + char_img.height + 70
    img = Image.new('RGBA', (image_width,image_height))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"{char_name.upper()}"
    label_fnt_size = 70
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    draw = ImageDraw.Draw(img)
    draw.text((10, char_img.height + 20), label_text, font=label_fnt, fill="black")

    img.paste(char_img, (18,10), mask=char_img)

    return img

In [11]:
!mkdir -p PagedOutArticle/surface_character_mini_diagrams

In [14]:
"""
level_character_set_images = [None]
for i in range(1,16):
    charset_ref = level_charsets[i]
    colors = level_colors[i]
    character_set = character_sets[charset_ref]
    character_images = {}
    for character_name in character_set:
        img = paintRawCharacter(character_set, character_name, colors)
        character_images["$"+character_name] = img
        if character_name:
            img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
            img.save(f"surface_characters/{i}_{character_name}.png")
    level_character_set_images += [character_images]
"""
from PIL import ImageDraw, ImageFont
for level in range(1,16):
    charset_ref = level_charsets[i]
    colors = level_colors[i]
    character_set = character_sets[charset_ref]
    for char in character_set:
        char_img = Image.open(f"surface_characters/{level}_{char}.png")
        img = generateMiniCharDiagram(char_img, char)
        img.save(f"PagedOutArticle/surface_character_mini_diagrams/{level}_{char}.png")


## Read in the Surface Structures Data (surfaceStructureData)

In [15]:
"""
Entries in this array have the structure, e.g.:

.BYTE $01  ; Number of Elements
.BYTE $01  ; Run-Length of element
.BYTE $20  ; Data (List of Character Set references)

"""
game_data_file = "game_data.asm"
lines_in_file = open(game_data_file,'r').readlines()

structure_data_header = lines_in_file[770]
assert ("surfaceStructureData" in structure_data_header)
structure_data_lines = lines_in_file[771:1230]

raw_surface_structure_data = [l[14:].strip() for l in structure_data_lines]
surface_structure_data = [x for l in raw_surface_structure_data for x in l.split(',')]

In [16]:
toInt = lambda x: int(x[1:],16)
surface_structures = []
i = 0
while (i < len(surface_structure_data)):
    num_elements = toInt(surface_structure_data[i])
    i += 1
    structure_data = []
    for j in range(0,num_elements):
        run_length = toInt(surface_structure_data[i])
        i += 1
        run_length = run_length & 0x1F
        raw_data = surface_structure_data[i:i+run_length]
        i += run_length
        data = [x[1:] for x in raw_data]
        structure_data += [data]
    surface_structures += [structure_data]

#for i,s in enumerate(surface_structures):
#    print(hex(i),s)

In [18]:
surface_structures[1]

[['21',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '22',
  '23']]

### Generate PNG Images of all the Structures

In [86]:
surface_strip_images = [None]
for j in range(1,16):
    character_images = level_character_set_images[j]
    strip_images = {}
    for i,surface_structure in enumerate(surface_structures):
        if not surface_structure:
            continue
        strip_max = max([len(x) for x in surface_structure])
        HEIGHT = 8 * strip_max
        WIDTH = len(surface_structure) * 8
        x,y = 0,HEIGHT-8
        img = Image.new( 'RGBA', (WIDTH,HEIGHT))
        for strip in surface_structure:
            for c in strip:
                char_image = character_images["$"+c]
                img.paste(char_image,(x,y),mask=char_image)
                y-=8
            x+=8
            y=HEIGHT-8
        ref_offset = i+1
        ref = ("00" + hex(ref_offset)[2:].upper())[-2:]
        strip_images[ref] = img

        img = img.resize((img.width * 5, img.height * 5), Image.NEAREST)
        #img.save(f"surface_structures/{j}_{i}.png")
        #img.save(f"surface_structures_hex/{j}_{ref}.png")
    surface_strip_images += [strip_images]

### Get All Chars in Each Structure

In [36]:
flatten = lambda l: [e for sublist in l for e in sublist]
surface_structure_chars = [None]
for i,surface_structure in enumerate(surface_structures):
    if not surface_structure:
        continue
    surface_structure_chars += [list(set(flatten(surface_structure)))]

### Read in the Dreadnought Data for All Levels

In [19]:
def getDreadnoughtData(level_data_lines):
    raw_dreadnought_data = [l[14:].strip() for l in level_data_lines]
    dreadnought_data = [x for l in raw_dreadnought_data for x in l.split(',')]

    end_of_base_data = dreadnought_data.index('$00')
    base_data = [x[1:] for x in dreadnought_data[:end_of_base_data]]
    raw_location_data = dreadnought_data[end_of_base_data+1:]
    location_data = []
    for i in range(0, len(raw_location_data), 3):
        hi_byte = int(raw_location_data[i][1:],16)
        if not hi_byte:
            break
        hi_byte |= 0x80
        hi_byte &= 0xBF
        hi_byte = hex(hi_byte)[2:]
        lo_byte = raw_location_data[i+1][1:]
        hex_location = hi_byte + lo_byte
        location = int(hex_location,16)
        structure = raw_location_data[i+2][1:]
        location_data += [(hex_location, location,structure)]
    dreadnought_surface = (base_data,location_data)
    return dreadnought_surface

# Read in bulk of the dreadnought data from level data first.
level_data_file = "uridium/src/level_data.asm"
lines_in_file = open(level_data_file,'r').readlines()

dreadnought_surfaces = {}
level_data_lines = []
current_dreadnought = ""
for l in lines_in_file:
    if l.startswith("level") and "DreadnoughtData" in l:
        if current_dreadnought:
            dreadnought_surfaces[current_dreadnought] = getDreadnoughtData(level_data_lines)
        current_dreadnought = l[:22].strip()
        level_data_lines = []
    if ".BYTE" not in l:
        continue
    level_data_lines += [l]
dreadnought_surfaces[current_dreadnought] = getDreadnoughtData(level_data_lines)

# Read in remaining dreadnoughts from charset.asm
level_data_file = "uridium/src/charset.asm"
lines_in_file = open(level_data_file,'r').readlines()

assert "level7DreadnoughtData" in lines_in_file[1297]
level_data_lines = [l for l in lines_in_file[1298:1358] if ".BYTE" in l]
dreadnought_surfaces["level7DreadnoughtData"] = getDreadnoughtData(level_data_lines)

assert "level8DreadnoughtData" in lines_in_file[1358]
level_data_lines = [l for l in lines_in_file[1359:1425] if ".BYTE" in l]
dreadnought_surfaces["level8DreadnoughtData"] = getDreadnoughtData(level_data_lines)


### Get the chars used in the surface each level

In [44]:
level_chars = [None]
for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set(base_data))
    chars = []
    for strip in strips:
        ref = int(strip,16)
        chars += [surface_structure_chars[ref]]
    level_chars += [list(set(flatten(chars)))]
len(level_chars[1])

44

### Diagram the Chars Used In the surface Each Level

In [114]:
import math

for LEVEL in range(1,16):
    chars = level_chars[LEVEL]
    height = 600
    width = 3900
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)
    
    x = 5
    y = 5
    for char in sorted(chars):
        strip_image = Image.open(f"PagedOutArticle/surface_character_mini_diagrams/{LEVEL}_{char}.png")
        img.paste(strip_image,(x,y),mask=strip_image)
        x += strip_image.width + 20
        if x > 3700:
            x = 5
            y += 250
    img.save(f"PagedOutArticle/surface_characters_{LEVEL}.png")



In [246]:
len(level_chars[1])

44

In [245]:
LEVEL = 1
chars = level_chars[LEVEL]
height = 250
width = 7800

img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x = 5
y = 5
for char in sorted(chars):
    strip_image = Image.open(f"PagedOutArticle/surface_character_mini_diagrams/{LEVEL}_{char}.png")
    img.paste(strip_image,(x,y),mask=strip_image)
    x += strip_image.width + 20
    if x > 7600:
        x = 5
        y += 250
img.save(f"PagedOutArticle/surface_characters_{LEVEL}.png")



### Get the chars used in the structures for each level

In [107]:
level_structure_chars = [None]
for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set([strip for _, _, strip in location_data]))
    chars = []
    for strip in strips:
        ref = int(strip,16)
        chars += [surface_structure_chars[ref]]
    level_structure_chars += [[x for x in list(set(flatten(chars))) if x not in level_chars[LEVEL]]]
len(level_structure_chars[1])

106

### Diagram the Chars Used In the Structures for Each Level

In [119]:
import math

for LEVEL in range(1,16):
    chars = [x for x in level_structure_chars[LEVEL] if x not in ['5A','61']]
    height = 750
    width = 6000
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)
    
    x = 5
    y = 5
    for char in sorted(chars):
        strip_image = Image.open(f"PagedOutArticle/surface_character_mini_diagrams/{LEVEL}_{char}.png")
        img.paste(strip_image,(x,y),mask=strip_image)
        x += strip_image.width + 20
        if x > 5900:
            x = 5
            y += 250
    img.save(f"PagedOutArticle/surface_characters_structures_{LEVEL}.png")



### Diagram the Strips Used In Each Level

In [45]:
!mkdir -p PagedOutArticle/surface_structures_for_level

In [163]:
import math

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set(base_data))

    strip_height = max([Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png").height for strip in strips])
    height = strip_height + 100
    width = sum([Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png").width + 50 for strip in strips])
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)

    x = 5
    for strip in strips:
        strip_image = Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png")
        y = strip_height - strip_image.height # Y position is relative to the bottom of the map.
        img.paste(strip_image,(x,y),mask=strip_image)

        label_text = f"{strip}"
        label_fnt_size = 42
        label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
        draw.text((x + int(strip_image.width/2) - 30, strip_height + 10), label_text, font=label_fnt, fill="black")

        x+= strip_image.width + 50
    img.save(f"PagedOutArticle/surface_structures_for_level/{LEVEL}.png")



In [204]:
LEVEL = 1
base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
strips = ['05', '5C','06','07','0C','08', '0E', '04', '0F', '0A', '20', '67', '09', '0B',  '02', '7A', '7B']
print(strips)
strip_height = max([Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png").height for strip in strips])
height = strip_height + 100
width = sum([Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png").width + 50 for strip in strips])

img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x = 5
for strip in strips:
    strip_image = Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png")
    y = strip_height - strip_image.height # Y position is relative to the bottom of the map.
    img.paste(strip_image,(x,y),mask=strip_image)

    label_text = f"{strip}"
    label_fnt_size = 42
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    draw.text((x + int(strip_image.width/2) - 30, strip_height + 10), label_text, font=label_fnt, fill="black")

    x+= strip_image.width + 50
img.save(f"PagedOutArticle/surface_structures_for_level/{LEVEL}.png")



['05', '5C', '06', '07', '0C', '08', '0E', '04', '0F', '0A', '20', '67', '09', '0B', '02', '7A', '7B']


In [205]:
import math

LEVEL = 1
strips = ['05', '5C','06','07','0C','08', '0E', '04', '0F', '0A', '20', '67', '09', '0B',  '02', '7A', '7B']

strip_height = max([Image.open(f"PagedOutArticle/surface_section_diagrams/{LEVEL}_{strip}_large_text.png").height for strip in strips])
height = strip_height + 100
width = sum([Image.open(f"PagedOutArticle/surface_section_diagrams/{LEVEL}_{strip}_large_text.png").width + 50 for strip in strips])

img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x = 5
for strip in strips:
    strip_image = Image.open(f"PagedOutArticle/surface_section_diagrams/{LEVEL}_{strip}_large_text.png")
    y = strip_height - strip_image.height # Y position is relative to the bottom of the map.
    img.paste(strip_image,(x,y),mask=strip_image)

    x+= strip_image.width + 50
img.save(f"PagedOutArticle/surface_structures_for_level/{LEVEL}_alt.png")



In [93]:
import math

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set(base_data))

    strip_height = max([Image.open(f"surface_character_set_diagrams/surface_section_diagrams/{LEVEL}_{strip}.png").height for strip in strips])
    height = strip_height + 100
    width = sum([Image.open(f"surface_character_set_diagrams/surface_section_diagrams/{LEVEL}_{strip}.png").width + 50 for strip in strips])
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)

    x = 5
    for strip in strips:
        strip_image = Image.open(f"surface_character_set_diagrams/surface_section_diagrams/{LEVEL}_{strip}_large_text.png")
        y = strip_height - strip_image.height # Y position is relative to the bottom of the map.
        img.paste(strip_image,(x,y),mask=strip_image)

        x+= strip_image.width + 50
    img.save(f"PagedOutArticle/surface_structures_for_level/{LEVEL}_alt.png")



### Diagram the Structures Used In Each Level

In [70]:
!mkdir -p PagedOutArticle/located_structures_for_level

In [91]:
import math

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set([strip for _, _, strip in location_data]))
    strips = [strip for strip in strips if strip not in ['5D','60','5E','64','89','8A','63','88','1E','16','56','14']]
    strip_height = max([Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png").height for strip in strips])
    height = 900
    width = 2300
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)

    x = 5
    y = 5
    for strip_count, strip in enumerate(strips):
        strip_image = Image.open(f"surface_structures_hex/{LEVEL}_{strip}.png")
        sy = strip_height - strip_image.height # Y position is relative to the bottom of the map.
        img.paste(strip_image,(x,y + sy),mask=strip_image)

        label_text = f"{strip}"
        label_fnt_size = 42
        label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
        draw.text((x + int(strip_image.width/2) - 30, y + strip_height + 10), label_text, font=label_fnt, fill="black")

        x+= strip_image.width + 50
        if x > 2100:
            x = 5
            y += 310
    img.save(f"PagedOutArticle/located_structures_for_level/{LEVEL}.png")



In [ ]:
['1F', '68', '63', '15', '88', '52', '54', '1D', '5F', '6B',
 '18', '19', '5E', '64', '14', '65', '11', '5D', '6A', '61',
 '0D', '13', '8A', '53', '1E', '16', '56', '69', '55', '89', '66', '60']

In [138]:
import math

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strips = list(set([strip for _, _, strip in location_data]))
    strips = [strip for strip in strips if strip not in ['5D','60','89']]
    print(strips)
    
    strip_height = max([Image.open(f"surface_character_set_diagrams/surface_structure_diagrams/{LEVEL}_{strip}.png").height for strip in strips])
    height = 5000
    width = 7800
    
    img = Image.new( 'RGBA', (width,height))
    draw = ImageDraw.Draw(img)

    x = 5
    y = 5
    for strip_count, strip in enumerate(strips):
        strip_image = Image.open(f"surface_character_set_diagrams/surface_structure_diagrams/{LEVEL}_{strip}.png")
        sy = strip_height - strip_image.height # Y position is relative to the bottom of the map.
        img.paste(strip_image,(x,y + sy),mask=strip_image)

        x+= strip_image.width + 50
        if x > 7300:
            x = 5
            y += strip_height
    img.save(f"PagedOutArticle/located_structures_for_level/{LEVEL}_alt.png")



['1F', '68', '63', '15', '88', '52', '54', '1D', '5F', '6B', '18', '19', '5E', '64', '14', '65', '11', '6A', '61', '0D', '13', '8A', '53', '1E', '16', '56', '69', '55', '66']
['10', '15', '68', '72', '52', '1D', '6B', '51', '19', '18', '6C', '14', '65', '11', '6A', '61', '5A', '0D', '13', '76', '7C', '53', '16', '79', '56', '69', '55', '7F', '78', '1A', '58', '54', '59']
['10', '1F', '15', '68', '52', '1D', '6B', '51', '19', '18', '14', '65', '11', '6A', '61', '0D', '13', '53', '1E', '16', '1C', '56', '69', '55', '54']
['10', '1F', '15', '63', '68', '52', '1D', '5F', '6B', '18', '19', '8B', '64', '14', '65', '11', '6A', '61', '0D', '13', '8A', '8C', '53', '1E', '56', '69', '55']
['1F', '15', '68', '52', '1D', '6B', '18', '51', '19', '6C', '8B', '70', '8D', '14', '6D', '65', '11', '86', '7E', '6A', '61', '75', '0D', '13', '91', '8E', '8C', '53', '1E', '16', '79', '56', '69', '55', '78', '54']
['10', '1F', '15', '63', '68', '52', '74', '1D', '6B', '18', '51', '19', '8B', '70', '8D', '64'

In [212]:
import math

# '56', '63', '64', ,'8A','61'
LEVEL = 1
strips = ['18', '1F', '68', '52', '19', '14', '65', '11', '6A',
          '13', '53','55', '1E', '15','1D',   '16', '69', '54', '66',
          '6B','0D','88',  '5E','5F'
         ]

strip_height = max([Image.open(f"surface_character_set_diagrams/surface_structure_diagrams/{LEVEL}_{strip}.png").height for strip in strips])
height = 2000
width = 6500

img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x = 5
y = 5
for strip_count, strip in enumerate(strips):
    strip_image = Image.open(f"surface_character_set_diagrams/surface_structure_diagrams/{LEVEL}_{strip}.png")
    sy = strip_height - strip_image.height # Y position is relative to the bottom of the map.
    img.paste(strip_image,(x,y + sy),mask=strip_image)

    x+= strip_image.width + 50
    if x > 6200:
        x = 5
        y += strip_height + 50
img.save(f"PagedOutArticle/located_structures_for_level/{LEVEL}_alt.png")



In [213]:
import math

# '56', '63', '64', ,'8A','61'
LEVEL = 1
strips = ['18', '1F', '68', '52', '19', '14', '65', '11', '6A',
          '13', '53','55', '1E', '15','1D',   '16', '69', '54', '66',
          '6B','0D','88',  '5E','5F'
         ]

strip_height = max([Image.open(f"PagedOutArticle/surface_structure_diagrams/{LEVEL}_{strip}.png").height for strip in strips])
height = 2000
width = 6500

img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x = 5
y = 5
for strip_count, strip in enumerate(strips):
    strip_image = Image.open(f"PagedOutArticle/surface_structure_diagrams/{LEVEL}_{strip}.png")
    sy = strip_height - strip_image.height # Y position is relative to the bottom of the map.
    img.paste(strip_image,(x,y + sy),mask=strip_image)

    x+= strip_image.width + 50
    if x > 6200:
        x = 5
        y += strip_height + 50
img.save(f"PagedOutArticle/located_structures_for_level/{LEVEL}_alt.png")



### Render the Levels with The Structures Only

In [84]:
!mkdir -p PagedOutArticle/surfaces_structures_only

In [87]:
import math

LOCATION_START = 0x8200
LOCATION_END = 0xA400
DREADNOUGHT_COLS = 0x200

width = DREADNOUGHT_COLS * 8 # 4096
height = int((LOCATION_END - LOCATION_START) / DREADNOUGHT_COLS) * 8 # 128

level_images = [None]
for LEVEL in range(1,16):
    img = Image.new( 'RGBA', (width,height))

    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strip_images = surface_strip_images[LEVEL]

    x,y = 512,0
    for hex_offset, location, structure in location_data:
        if structure == "20":
            print(structure)
        if structure not in strip_images:
            continue
        strip_image = strip_images[structure]
        offset = location - LOCATION_START
        x = (offset % DREADNOUGHT_COLS) * 8
        # The location given is for the bottom of the object (uridium draws upwards)
        y = (int(math.ceil(offset / DREADNOUGHT_COLS)) * 8) - strip_image.height
        img.paste(strip_image,(x,y),mask=strip_image)
    img.save(f"PagedOutArticle/surfaces_structures_only/{LEVEL}_full.png")
    if LEVEL == 12:
        img = img.crop((350,0,img.width-470,img.height))
    elif LEVEL == 15:
        img = img.crop((410,0,img.width-200,img.height))
    elif LEVEL == 9:
        img = img.crop((480,0,img.width-300,img.height))
    elif LEVEL == 2:
        img = img.crop((480,0,img.width-400,img.height))
    elif LEVEL == 10:
        img = img.crop((480,0,img.width-430,img.height))
    elif LEVEL == 13:
        img = img.crop((480,0,img.width-430,img.height))
    else:
        img = img.crop((480,0,img.width-450,img.height))
    #img = img.resize((img.width * 5, img.height * 5), Image.NEAREST)
    img.save(f"PagedOutArticle/surfaces_structures_only/{LEVEL}.png")
    level_images += [img]

### Get the Base Data for Each Dreadnought

In [188]:
for LEVEL in range(1,16):
    img = Image.new( 'RGBA', (width,height))

    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    print(".BYTE " + ',$'.join(base_data[:33]))


.BYTE 05,$5C,$5C,$06,$06,$06,$06,$5C,$5C,$07,$0C,$08,$08,$08,$08,$08,$08,$08,$08,$08,$08,$05,$5C,$09,$0A,$0A,$0A,$67,$5C,$5C,$5C,$09,$0A
.BYTE 05,$5C,$5C,$02,$5C,$5C,$5C,$5C,$04,$0C,$02,$5C,$5C,$5C,$5C,$5C,$06,$06,$06,$06,$06,$06,$06,$06,$03,$06,$06,$06,$06,$06,$06,$04,$0C
.BYTE 05,$5C,$5C,$04,$02,$5C,$06,$5C,$5C,$5C,$06,$5C,$07,$0C,$08,$08,$08,$08,$08,$08,$02,$5C,$04,$0C,$08,$08,$08,$08,$08,$08,$02,$5C,$06
.BYTE 20,$20,$20,$20,$05,$5C,$5C,$04,$7A,$02,$5C,$06,$06,$06,$06,$06,$5C,$07,$20,$20,$20,$20,$0C,$08,$08,$08,$08,$08,$08,$08,$20,$20,$20
.BYTE 05,$5C,$5C,$5C,$04,$05,$5C,$06,$06,$06,$06,$04,$7A,$02,$5C,$04,$7A,$02,$5C,$04,$7A,$02,$5C,$06,$06,$06,$5C,$5C,$5C,$5C,$09,$0A,$67
.BYTE 20,$20,$20,$20,$05,$04,$20,$20,$20,$20,$05,$04,$7A,$7B,$7B,$7B,$02,$5C,$06,$06,$06,$5C,$04,$7A,$7B,$7B,$7B,$02,$07,$20,$20,$20,$20
.BYTE 20,$20,$20,$20,$05,$5C,$5C,$5C,$09,$0B,$05,$5C,$04,$0E,$0F,$0F,$05,$5C,$06,$06,$06,$06,$06,$5C,$07,$0E,$0F,$0F,$02,$5C,$04,$0E,$0F
.BYTE 20,$20,$20,$20,$05,$07,$20,$20,$20,

###  Build up a partial level only  - just the surface

In [218]:
test_image = Image.open(f"surface_character_set_diagrams/character_sets/1_00.png")
character_height = test_image.height
character_width = test_image.width

width = DREADNOUGHT_COLS * character_width # padding from diagram
height = int((LOCATION_END - LOCATION_START) / DREADNOUGHT_COLS) * character_height # 128

LEVEL = 1
base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x,y = 64 * character_width,0
for structure in base_data[:33]:
    strip_image = Image.open(f"surface_character_set_diagrams/surface_strips_hex/{LEVEL}_{structure}_no_text.png")
    # Y position is relative to the bottom of the map.
    y = height - strip_image.height
    img.paste(strip_image,(x,y),mask=strip_image)
    draw.rectangle((x, 0, x+strip_image.width-1, strip_image.height+2), fill=None, outline="black",width=2)

    x+= strip_image.width

img = img.resize((int(img.width * .25), int(img.height * .25)), Image.NEAREST)
img = img.crop((1750,0,img.width-10470,img.height))
img.save(f"PagedOutArticle/{LEVEL}_partial.png")


###  Build up a partial level only  - surface and structures

In [239]:
test_image = Image.open(f"surface_character_set_diagrams/character_sets/1_00.png")
character_height = test_image.height
character_width = test_image.width

width = DREADNOUGHT_COLS * character_width # padding from diagram
height = int((LOCATION_END - LOCATION_START) / DREADNOUGHT_COLS) * character_height # 128

LEVEL = 1
base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
img = Image.new( 'RGBA', (width,height))
draw = ImageDraw.Draw(img)

x,y = 64 * character_width,0
for structure in base_data[:33]:
    strip_image = Image.open(f"surface_character_set_diagrams/surface_strips_hex/{LEVEL}_{structure}_no_text.png")
    # Y position is relative to the bottom of the map.
    y = height - strip_image.height
    img.paste(strip_image,(x,y),mask=strip_image)

    x+= strip_image.width

for hex_offset, location, structure in location_data:
    print(hex_offset, structure)
    strip_image = Image.open(f"PagedOutArticle/surface_structure_diagrams/{LEVEL}_{structure}.png")

    offset = location - LOCATION_START
    x = (offset % DREADNOUGHT_COLS) * character_width

    # The location given is for the bottom of the object (uridium draws upwards)
    y = (int(math.ceil(offset / DREADNOUGHT_COLS)) * character_height) - (strip_image.height - 10) # padding at bottom of diagram
    img.paste(strip_image,(x,y),mask=strip_image)


img = img.resize((int(img.width * .25), int(img.height * .25)), Image.NEAREST)
img = img.crop((1750,0,img.width-10890,img.height))
img.save(f"PagedOutArticle/{LEVEL}_partial_structures.png")


9844 0D
8e4C 14
9458 1F
945B 1F
9a5B 1F
a05B 13
945E 13
8a7E 11
8a83 11
8a88 11
8a8D 11
8a92 11
8a97 11
8a9C 11
8aA1 53
9e7E 11
9e83 11
9e88 11
9e8D 11
9e92 11
9e97 11
9e9C 11
9eA1 53
947C 55
9688 54
988E 54
8a81 19
8a8E 19
8a97 19
8a9C 19
9e80 19
9e8A 19
9e91 19
9eB5 5D
9eB6 60
96C9 66
a0D5 65
8eD8 52
8eDB 1F
8eDE 1F
8eE1 1F
a0D9 61
96DA 56
a0E2 61
a0DD 68
8909 1D
a309 1E
8917 1D
a317 1E
8925 1D
a325 1E
9537 14
9f3D 5D
9f3E 64
873E 63
9f46 60
9d3E 18
9d44 18
8b44 18
8b3E 18
8646 18
a046 18
9541 52
9770 15
978A 15
9779 69
9782 69
8b6F 11
8b74 11
8b79 11
8b7E 11
8b83 11
8b88 53
9f6F 11
9f74 11
9f79 11
9f7E 11
9f83 11
9f88 53
8b72 19
8b7A 19
8b83 19
8b89 19
9f70 19
9f79 19
9f86 19
899E 1D
8f9E 16
8b97 65
8f96 56
a1A2 5E
a1A4 64
a1AC 64
95A8 6B
a1B4 5E
a1B6 5E
a1B8 5E
95A3 88
a1BA 5F
89A4 63
89AC 63
95B3 89
a1A0 5E
a19E 5E
a19C 5E
a19A 5E
a198 5E
9d98 6A
9d9D 11
9dA2 11
9dA7 11
9dAA 11
9dAF 11
9dB4 11
9dB8 53
8dB8 14
a197 8A


## Surface Section Diagrams

In [202]:
def generateSurfaceSectionDiagramLargeText(section_image, section_name, level_name, level_number):

    image_width = 100 + section_image.width + 5
    image_height = 10 + section_image.height + 10
    img = Image.new('RGBA', (image_width,image_height))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(1,1), tuple(x-1 for x in img.size)], outline="black")

    # Sprite label
    label_text = f"URIDIUM/{level_name.upper()}/{section_name.upper()}"
    label_fnt_size = 80
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (10, (img.height - (label.height + 20))))

    # Main Sprite image
    #character_image = character_image.resize((character_image.width * 2, character_image.height * 2), Image.NEAREST)
    img.paste(section_image, (100,10))

    return img

In [201]:
!mkdir -p PagedOutArticle/surface_section_diagrams

In [203]:
import math
from PIL import Image, ImageColor, ImageDraw, ImageFont

LOCATION_START = 0x8200
LOCATION_END = 0xA400
DREADNOUGHT_COLS = 0x200

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    level_name = level_names[LEVEL]
    for structure in base_data:
        strip_image = Image.open(f"surface_character_set_diagrams/surface_strips_hex/{LEVEL}_{structure}.png")
        section_diagram = generateSurfaceSectionDiagramLargeText(strip_image, structure, level_name, LEVEL)
        section_diagram.save(f"PagedOutArticle/surface_section_diagrams/{LEVEL}_{structure}_large_text.png")


## Surface Structure Diagrams

In [210]:
def generateSurfaceStructureDiagram(section_image, section_name, level_name, level_number):

    image_width = 5 + section_image.width + 5
    image_height = 80 + section_image.height + 10
    img = Image.new('RGBA', (image_width,image_height))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(1,1), tuple(x-1 for x in img.size)], outline="black")

    level_number_string = ('00'+str(level_number))[-2:]
    # Sprite label
    label_text = f"{section_name.upper()}"
    label_fnt_size = 70
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt
    img.paste(label, (5, 10))

    # Main Sprite image
    #character_image = character_image.resize((character_image.width * 2, character_image.height * 2), Image.NEAREST)
    img.paste(section_image, (5,80))
    return img

In [191]:
!mkdir -p PagedOutArticle/surface_structure_diagrams/

In [211]:
import math
from PIL import Image, ImageColor, ImageDraw, ImageFont

LOCATION_START = 0x8200
LOCATION_END = 0xA400
DREADNOUGHT_COLS = 0x200

for LEVEL in range(1,16):
    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    level_name = level_names[LEVEL]
    for _, _, structure in location_data:
        strip_image = Image.open(f"surface_character_set_diagrams/surface_structures_hex/{LEVEL}_{structure}.png")
        section_diagram = generateSurfaceStructureDiagram(strip_image, structure, level_name, LEVEL)
        section_diagram.save(f"PagedOutArticle/surface_structure_diagrams/{LEVEL}_{structure}.png")


# Scratchpad